In [2]:
import torch

In [4]:
ratings = torch.tensor([
    [1, 1, 0, 1, 0, 0],
    [0, 0, 1, 1, 1, 0],
    [1, 0, 0, 0, 1, 1],
], dtype=torch.float32)

num_users, num_items = ratings.shape

type(num_users), type(num_items)

(int, int)

In [5]:
test_items = {}

for user_id in range(num_users):
  positive_items = torch.where(ratings[user_id] == 1)[0]

  test_items[user_id] = positive_items[-1].item()

In [6]:
train_positive = []

for user_id in range(num_users):
  positive_items = torch.where(ratings[user_id] == 1)[0]

  for item_id in positive_items:
    item_id = item_id.item()

    # 테스트 데이터에 쓸 user_id라면 제외해주기
    if item_id == test_items[user_id]:
      continue

    # 학습할 관측된 값의 인덱스
    train_positive.append((user_id, item_id))

In [23]:
users = []
items = []
labels = []

num_negatives = 2

# train_positive는 test할 각각의 users에 대해서 하나의 item을 제외한 
# 모든 [(user_id, item_id), ...]
for user_id, positive_item in train_positive:
  # 일단 하나의 positive 추가해주기
  users.append(user_id)
  items.append(positive_item)
  labels.append(1.0)

  # 사용자의 모든 관측되지 않은 값 확인하기
  negative_candidates = torch.where(ratings[user_id] == 0)[0]
  indices = torch.randperm(len(negative_candidates))[:num_negatives]
  sampled_negatives = negatives = negative_candidates[indices]

  for negative_item in sampled_negatives:
    users.append(user_id)
    items.append(negative_item)
    labels.append(0.0)

users = torch.tensor(users, dtype=torch.long)
items = torch.tensor(items, dtype=torch.long)
labels = torch.tensor(labels, dtype=torch.float32)

In [16]:
users.shape, items.shape, labels.shape

(torch.Size([18]), torch.Size([18]), torch.Size([18]))

In [17]:
num_users, num_items, len(labels)

(3, 6, 18)

In [24]:
from torch import nn

class Recommender(nn.Module):
  def __init__(self, num_users, num_items, embedding_dim=4):
    super().__init__()

    self.user_embedding = nn.Embedding(num_users, embedding_dim)
    self.item_embedding = nn.Embedding(num_items, embedding_dim)

  def forward(self, user_ids, item_ids):
    users_vec = self.user_embedding(user_ids)
    items_vec = self.item_embedding(item_ids)

    return (users_vec * items_vec).sum(dim=1)

In [27]:
model = Recommender(num_users, num_items)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

criterion = torch.nn.BCEWithLogitsLoss()

for epoch in range(200):
  # recommender (model)의 기울기 최적화
  optimizer.zero_grad()

  # model에서 벡터 유사도 측정 결과 만들기
  scores = model(users, items)

  # 그 결과 -> sigmoid -> 1, 0 비교 손실함수
  loss = criterion(scores, labels)

  # 역전파를 통해 grad 누적
  loss.backward()

  # 기울기 업데이트해주기
  optimizer.step()

  # 에포크들마다 찍어주기
  if epoch % 20 == 0:
    print(f"{epoch:3d}, {loss.item():.4f}")

  0, 0.7550
 20, 0.5277
 40, 0.3660
 60, 0.2374
 80, 0.1364
100, 0.0727
120, 0.0411
140, 0.0262
160, 0.0183
180, 0.0136


In [ ]:
# 사용자가 특정 상품에 대해서 선호할지 맞추는 것에 대해서 알아보기
# test는 {user_id: item_id} 형태로 되어있음. 
user_id = 0

# [[user_ids], [item_ids]] 배열 2개 만들어주기

# user_ids 배열
user_tensor = torch.full(
  (num_items,),
  user_id,
  dtype=torch.long
)

# 아이템의 모든 클래스 뽑아주기 ()
item_tensor = torch.arange(num_items)

with torch.no_grad():
  scores = model(user_tensor, item_tensor)

torch.int64

In [31]:
seen_items = torch.where(ratings[user_id] == 1)[0].tolist()

test_item = test_items[user_id]

seen_items.remove(test_item)

scores[seen_items] = -float("inf")

In [35]:
test_item

3

In [ ]:
# logits에서 top_k 뽑아주기
top_scores, top_items = torch.topk(scores, k=3)

print(top_items)

tensor([5, 3, 2])


In [33]:
import math

def ndcg_at_k(top_items, target_item, k):
  top_items = top_items[:k].tolist()

  if target_item not in top_items:
    return 0.0

  rank = top_items.index(target_item) + 1

  return 1 / math.log2(rank+1)

In [34]:
print(ndcg_at_k(top_items, test_items[user_id], 3))

0.6309297535714575
